# Part 2: Computer Vision Problem Formulation and CNN Prototype
## Manufacturing Defect Classification using Convolutional Neural Network

**Dataset:** Synthetic Manufacturing Defect Image Dataset  
**Classes:** `dent`, `normal`, `scratch`, `stain`  
**Problem Type:** Multi-class Image Classification (4 classes)

## Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                      Dense, Dropout, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
tf.get_logger().setLevel('ERROR')

os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)

print(f'TensorFlow version: {tf.__version__}')
print('All libraries loaded!')

---
## Task 1: Problem Identification

**Selected Problem Type: Image Classification**

The dataset consists of product surface images organized into 4 folders, each representing a distinct surface condition: `normal`, `scratch`, `dent`, and `stain`. Each image belongs to exactly **one** class with no spatial localization or segmentation boundary required.

This makes it a **multi-class image classification** problem — the model receives an image as input and outputs a single label from the 4 possible classes. This is the appropriate formulation because:

- Each image represents a whole product surface (no need to locate specific regions)
- Labels are mutually exclusive (one image = one defect type)
- There are no bounding box annotations or pixel-level masks (ruling out detection/segmentation)

**Real-world application:** Automated quality inspection in manufacturing — a camera on the production line captures product images, and the CNN classifies each item as defective or not, reducing reliance on manual inspection.

---
## Task 2: Dataset Exploration

In [ ]:
classes  = ['dent', 'normal', 'scratch', 'stain']
base     = 'images'  # folder containing class subfolders
IMG_SIZE = 64

# Count images per class
class_counts = {cls: len([f for f in os.listdir(f'{base}/{cls}') if f.endswith('.png')])
                for cls in classes}

print('=== Dataset Summary ===')
print(f'Number of classes     : {len(classes)}')
print(f'Classes               : {classes}')
print(f'Images per class      : {class_counts}')
print(f'Total images          : {sum(class_counts.values())}')

# Check image dimensions
sample_path = f'{base}/normal/{os.listdir(f"{base}/normal")[0]}'
sample_img  = Image.open(sample_path)
print(f'Image dimensions      : {sample_img.size[0]}x{sample_img.size[1]} pixels')
print(f'Image mode            : {sample_img.mode}')
print(f'Dataset balanced      : {len(set(class_counts.values())) == 1}')

In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#F44336', '#4CAF50', '#FF9800', '#2196F3']
bars = axes[0].bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black')
for bar, v in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1, str(v), ha='center', fontweight='bold')
axes[0].set_title('Images per Class', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(class_counts.values(), labels=class_counts.keys(), colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution', fontweight='bold')

plt.suptitle('Dataset Overview — Manufacturing Defect Dataset', fontweight='bold')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Sample images grid (2 per class)
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for col, cls in enumerate(classes):
    imgs = sorted(os.listdir(f'{base}/{cls}'))[:2]
    for row, img_file in enumerate(imgs):
        img = Image.open(f'{base}/{cls}/{img_file}')
        axes[row][col].imshow(img)
        axes[row][col].set_title(f'{cls.capitalize()}\n{img.size[0]}x{img.size[1]} RGB', fontsize=9)
        axes[row][col].axis('off')
plt.suptitle('Sample Images per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: The dataset is perfectly balanced — 120 images per class.')
print('No class imbalance handling needed.')

---
## Task 3: Image Preprocessing

In [ ]:
# Load all images
X, y = [], []
for label, cls in enumerate(classes):
    for fname in sorted(os.listdir(f'{base}/{cls}')):
        if fname.endswith('.png'):
            img = Image.open(f'{base}/{cls}/{fname}').convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            X.append(np.array(img))
            y.append(label)

# Convert to numpy arrays and normalize pixel values to [0, 1]
X = np.array(X, dtype='float32') / 255.0
y = np.array(y)

print(f'Image array shape : {X.shape}')   # (480, 64, 64, 3)
print(f'Label array shape : {y.shape}')   # (480,)
print(f'Pixel value range : [{X.min():.2f}, {X.max():.2f}]')
print(f'Target size       : {IMG_SIZE}x{IMG_SIZE} RGB')

In [ ]:
# Train / Validation / Test split (70% / 12% / 18% approx)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# One-hot encode labels for multi-class classification
y_train_cat = to_categorical(y_train, 4)
y_val_cat   = to_categorical(y_val,   4)
y_test_cat  = to_categorical(y_test,  4)

print(f'Train : {X_train.shape} — {y_train.shape}')
print(f'Val   : {X_val.shape}   — {y_val.shape}')
print(f'Test  : {X_test.shape}  — {y_test.shape}')

# Data Augmentation to improve generalization
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train)
print('\nData augmentation configured: rotation, shift, flip, zoom')

**Preprocessing Steps:**
- All images resized to 64×64 pixels (consistent input shape for the CNN).
- Pixel values normalized from [0, 255] → [0.0, 1.0] (helps gradient stability).
- Stratified 80/20 train-test split, then 15% of train reserved for validation.
- Data augmentation applied on training data only to improve generalization on small dataset.

---
## Task 4: CNN Model Creation

In [ ]:
model = Sequential([
    # ── Block 1 ─────────────────────────────────────
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMG_SIZE, IMG_SIZE, 3), name='conv1'),
    BatchNormalization(),
    MaxPooling2D(2, 2, name='pool1'),          # 64x64 → 32x32

    # ── Block 2 ─────────────────────────────────────
    Conv2D(64, (3,3), activation='relu', padding='same', name='conv2'),
    BatchNormalization(),
    MaxPooling2D(2, 2, name='pool2'),          # 32x32 → 16x16

    # ── Block 3 ─────────────────────────────────────
    Conv2D(128, (3,3), activation='relu', padding='same', name='conv3'),
    BatchNormalization(),
    MaxPooling2D(2, 2, name='pool3'),          # 16x16 → 8x8

    # ── Classifier ──────────────────────────────────
    Flatten(name='flatten'),
    Dense(128, activation='relu', name='dense1'),
    Dropout(0.4),
    Dense(4, activation='softmax', name='output')  # 4 classes
], name='ManufacturingDefectCNN')

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Model Architecture Explanation:**

| Layer | Purpose |
|---|---|
| `Conv2D(32)` | Learns 32 low-level feature maps (edges, textures) from 3x3 patches |
| `BatchNormalization` | Normalizes activations for training stability |
| `MaxPooling2D` | Reduces spatial dimensions by 2x, retaining dominant features |
| `Conv2D(64)` | Learns 64 mid-level features (patterns, shapes) |
| `Conv2D(128)` | Learns 128 high-level abstract features (defect-specific representations) |
| `Flatten` | Converts 2D feature maps to 1D vector for classification |
| `Dense(128, ReLU)` | Fully-connected layer combining learned features |
| `Dropout(0.4)` | Randomly disables 40% of neurons to prevent overfitting |
| `Dense(4, Softmax)` | Output layer — produces probability distribution over 4 classes |

---
## Task 5: Model Training and Evaluation

In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=32),
    validation_data=(X_val, y_val_cat),
    epochs=40,
    verbose=1
)

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
y_pred_prob = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.1f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'],     label='Train Loss', color='#1565C0')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#F44336', linestyle='--')
axes[0].set_title('Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='#1565C0')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#F44336', linestyle='--')
axes[1].set_title('Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.suptitle('CNN Training — Manufacturing Defect Classification', fontweight='bold')
plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=classes, yticklabels=classes)
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()

print('Confusion matrix saved!')

In [ ]:
# Sample predictions — one image per class
indices = [np.where(y_test == c)[0][0] for c in range(4)]
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for ax, idx in zip(axes, indices):
    img       = X_test[idx]
    true_lbl  = classes[y_test[idx]]
    pred_lbl  = classes[y_pred[idx]]
    conf      = y_pred_prob[idx][y_pred[idx]]
    color     = '#4CAF50' if true_lbl == pred_lbl else '#F44336'
    ax.imshow(img)
    ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf*100:.1f}%)',
                 color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=150, bbox_inches='tight')
plt.show()

**Result Interpretation:**

- The model achieves ~61% test accuracy on a 4-class balanced dataset (random baseline = 25%).
- `stain` is classified well (high precision/recall) — likely because stains are colorful and visually distinct.
- `dent` is the hardest class — its circular marks are subtle and may resemble normal surface textures.
- `normal` has high recall but lower precision — the model defaults to predicting normal when uncertain.
- Training accuracy (~96%) greatly exceeds validation accuracy, indicating **overfitting** on the small dataset (480 images).
- **Improvement strategies:** Transfer learning (e.g. MobileNet), more data, stronger augmentation.

---
## Task 6: CNN Concept Explanation

### What is Convolution?
Convolution is the core operation in a CNN. A small filter (e.g. 3×3 pixels) slides across the image and computes a dot product at each position. Each filter learns to detect a specific pattern — one might detect horizontal edges, another detects diagonal scratches, another detects circular shapes (like dents). The result is a **feature map** showing where that pattern appears in the image. By stacking multiple filters, the network builds a rich representation of the visual content.

### Why is Pooling Used?
Pooling (specifically Max Pooling) reduces the spatial size of feature maps by taking the maximum value in each small region. This serves two purposes: it reduces computation and parameters (making training faster), and it introduces **spatial invariance** — the model becomes less sensitive to the exact position of a feature. A scratch slightly shifted left or right is still recognized as a scratch.

### Why is ReLU Commonly Used in CNNs?
ReLU (Rectified Linear Unit) outputs `max(0, x)` — it passes positive values unchanged and sets negatives to zero. It is preferred because: (1) it is computationally cheap, (2) it avoids the vanishing gradient problem that affects sigmoid/tanh in deep networks, and (3) it introduces sparsity (many neurons output 0), which acts as a natural regularizer and speeds up learning.

### Why are CNNs Better than Feed-Forward Networks for Images?

| Property | Feed-Forward (Dense) | CNN |
|---|---|---|
| Input handling | Flattens image — loses spatial structure | Preserves 2D spatial relationships |
| Parameters | Huge (64×64×3 = 12,288 inputs × many neurons) | Shared weights via filters — far fewer parameters |
| Spatial patterns | Cannot learn position-aware patterns | Explicitly learns local patterns via convolution |
| Translation invariance | None — moving a feature changes the input | Pooling provides invariance to small shifts |
| Scalability | Impractical for large images | Scales efficiently with deeper filter hierarchies |

CNNs exploit the **local structure** and **translation invariance** of images — properties that dense networks cannot capture.

---
## Task 7: Business Use Case Mapping — Manufacturing Quality Control

**Domain:** Manufacturing / Industrial Quality Inspection

**Problem:** Manual visual inspection of products on a production line is slow, costly, and inconsistent — human fatigue leads to missed defects and false rejections.

**CNN Solution:** Mount cameras at inspection stations on the production line. As each product passes, a high-resolution image is captured and fed into a deployed CNN model (similar to the one built here). The model classifies the product in milliseconds as `normal`, `scratch`, `dent`, or `stain`.

**Business Impact:**
- **Speed:** CNN inspects thousands of items per hour vs. a few hundred manually.
- **Consistency:** No fatigue — the model's accuracy stays constant across shifts.
- **Cost reduction:** Fewer human inspectors needed; fewer defective products reaching customers.
- **Traceability:** Every prediction is logged with confidence scores for quality audits.
- **Actionable alerts:** Defective products (scratch, dent, stain) are automatically rejected; normal products proceed to packaging.

**Example companies using this:** BMW, Samsung, and Foxconn use CNN-based visual inspection systems in their factories to detect surface defects on car parts, circuit boards, and consumer electronics.